In [1]:
import pandas as pd

trees = pd.read_csv('trees_filtered_catarina.csv', sep=';', decimal=',')
trees.head()

,plot_id,tree_id,d_cm,pap_cm,ct_mm,ht_m,hcb_m,yd,papi,di,papj,dj,papk,cti,ctj,ctk,species,observations
0,10,1,NaN,48.4,NaN,5.1,1.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sb,Never debarked
1,10,2,46.7,NaN,22.0,10.4,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sb,NaN
2,10,3,63.8,NaN,25.0,10.0,3.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sb,Damaged bark
3,10,4,NaN,42.5,NaN,5.4,1.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sb,Never debarked
4,10,5,NaN,65.5,NaN,6.9,1.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sb,Never debarked


In [2]:
import numpy as np

mask = trees['d_cm'].isna() & trees['pap_cm'].notna()
trees.loc[mask, 'd_cm'] = trees.loc[mask, 'pap_cm'] / np.pi
print(f"Filled {mask.sum()} d_cm values from pap_cm")
trees[['plot_id','tree_id','d_cm','pap_cm']].head()

Filled 106 d_cm values from pap_cm


,plot_id,tree_id,d_cm,pap_cm
0,10,1,15.406198,48.4
1,10,2,46.700000,NaN
2,10,3,63.800000,NaN
3,10,4,13.528170,42.5
4,10,5,20.849298,65.5


Note: the below is then taken out of here and the bi and tri forks are manually calculated to create the file of the same name. 

In [3]:
trees.to_csv('lizirias_fieldwork_20260707.csv', index=False)
print("Saved trees dataframe to lizirias_fieldwork_20260707.csv")

Saved trees dataframe to lizirias_fieldwork_20260707.csv


In [4]:
import geopandas as gpd
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np

def calculate_nearest_neighbors(input_file, output_file, distance_m=30):
    # 1. Read the GeoDataFrame
    # Ensure the data is in a projected CRS (meters) before running this
    gdf = gpd.read_file(input_file)
    
    # CRITICAL CHECK: If your data is in Lat/Lon, reproject it first
    # Example: gdf = gdf.to_crs(epsg=32633)  # Replace 32633 with your UTM zone
    if gdf.crs.is_geographic:
        print("Warning: Input data is in geographic coordinates (Lat/Lon). Reprojecting to UTM...")
        # Auto-reproject to the best UTM zone
        gdf = gdf.to_crs(epsg=gdf.estimate_utm_crs())
        
    print(f"Reprojected CRS: {gdf.crs}")
    print(f"Total points: {len(gdf)}")

    # 2. Extract coordinates
    # We use the geometry's x and y coordinates
    coords = np.column_stack([gdf.geometry.x.values, gdf.geometry.y.values])

    # 3. Build a KDTree for fast spatial search
    # KDTree is much faster than looping for large datasets
    tree = cKDTree(coords)

    # 4. Query the tree
    # query() returns distances and indices
    # distance_upper_bound: the maximum distance (30 meters)
    # k: we don't limit the number of neighbors, we just get all within the distance
    # We set k=len(gdf) + 1 to ensure we get all possible matches
    distances, indices = tree.query(coords, k=len(gdf) + 1, distance_upper_bound=distance_m)

    # 5. Calculate counts
    # 'indices' contains the row numbers of the points found. 
    # If a point is not found, the index is len(gdf) (out of bounds).
    # We count how many indices are valid (less than len(gdf))
    
    # Create an array of 1s for each point, then mask out the "out of bounds" ones
    counts = np.sum(indices < len(gdf), axis=1)

    # 6. Adjust for self-counting (optional)
    # The query includes the point itself (distance = 0). 
    # If you want "neighbors excluding self", subtract 1.
    # counts = counts - 1 
    
    # 7. Add the new column to the GeoDataFrame
    gdf['point_count_30m'] = counts

    # 8. Save the result
    gdf.to_file(output_file, driver='GPKG') # Save as GeoPackage
    print(f"Results saved to {output_file}")
    
    # Display summary
    print(f"Average points within 30m: {gdf['point_count_30m'].mean():.2f}")
    print(f"Max points within 30m: {gdf['point_count_30m'].max()}")

# Usage
calculate_nearest_neighbors('heights_vector.gpkg', 'output_points_1.gpkg', distance_m=30)

Reprojected CRS: EPSG:3763
Total points: 2821
Results saved to output_points_1.gpkg
Average points within 30m: 16.91
Max points within 30m: 33


Then used 'join attributes by location (summary)' in QGIS to get the tallest point in each polygon